# Training GNN


In [ ]:
import pandas as pd 

from graph_network import build_graph
from ml import train_model
from plots import plot_loss, plot_rmse, plot_penalty, plot_training_gap

SIM_NAME = "example-model"

### Build Graph Architecture

In [2]:
# Build graph architecture
build_graph(SIM_NAME)

Loading first .vtu for case0
Building node ordering from CSV for case0
Building boundary flags for case0
Building edges and attributes for case0
Aligning timeseries for case0
Saving artifacts for case0


'/Volumes/connor/dev/projects/ns2d-surrogate/sims/example-model/training_data/graph/case0'

### Train GNN Model

In [3]:
# Train GNN model
CASE_NAME = 'case0'
r = train_model(sim_name=SIM_NAME, case_name=CASE_NAME, epochs=25, lr=1e-3,
                hidden_features=128, dropout_p=0.0, step_stride=5, print_state=False,
                lambda_div=1e-3, lambda_bnd=1e-2)
                
df = pd.DataFrame(r)
df.tail()

100%|██████████| 25/25 [00:46<00:00,  1.87s/it]

Saved model checkpoint to: /Volumes/connor/dev/projects/ns2d-surrogate/sims/example-model/training_data/graph/case0/model_graphsage.pt


,epoch,train_loss,val_loss,val_rmse_du,val_rmse_dv,val_rmse_dp,val_div,val_bnd
20,21,0.012048,0.000175,0.003605,0.003429,0.012232,0.000209,0.000051
21,22,0.011846,0.000138,0.003691,0.003753,0.010457,0.000209,0.000041
22,23,0.011772,0.000143,0.003436,0.003232,0.010955,0.000209,0.000037
23,24,0.011396,0.000162,0.008170,0.004868,0.008412,0.000209,0.000050
24,25,0.011424,0.000219,0.004387,0.004259,0.013426,0.000208,0.000067


### Validation

In [4]:
'''
Plot training and validation loss over epochs.

This chart visualises the progression of training and validation loss through 
epochs. The training loss indicates model fit to the training set, while 
validation loss shows generalisation to unseen data. A small generalisation gap 
(train_loss ~ val_loss) signals good generalisation; a widening gap suggests 
overfitting. The best epoch—corresponding to the minimum validation loss—is 
highlighted as the preferable model selection point.
'''

fig1 = plot_loss(df)
fig1.show()

In [5]:
'''
Plot validation RMSE for du, dv, dp over epochs.

This shows how the model's accuracy on predicting the changes in u-velocity, 
v-velocity, and pressure evolves during training. Each line corresponds to the 
RMSE of a specific component (du, dv, dp) for the validation set. Use this to 
diagnose which output (u, v, or p) the model finds most difficult to predict—if
a particular RMSE remains high or decreases slowly, it may require model or 
data adjustment. The vertical dashed line marks the epoch with the minimum 
validation loss (best epoch).
'''
fig2 = plot_rmse(df)
fig2.show()

In [6]:
'''
Plot validation divergence and boundary penalties over epochs.

This chart visualises how well the model enforces physical constraints during 
training. The divergence penalty (val_div) assesses how closely the predicted 
velocity field is incompressible (i.e., zero divergence). The boundary penalty 
(val_bnd) measures how well boundary conditions are satisfied in the model's 
predictions. Ideally, both penalties should decrease as the model learns; 
consistently high or rising values indicate the model struggles to respect 
physical laws. Consider adjusting λ_div/λ_bnd or revisiting the model 
architecture if divergence or boundary penalty is unacceptably high.
'''
fig3 = plot_penalty(df)
fig3.show()


In [7]:
'''
Plot the generalisation gap and mean validation RMSE over epochs.

This chart visualises two key aspects:
- The generalisation gap (train_loss - val_loss), indicating the difference in 
  model performance between training and validation data. A small or near-zero 
  gap suggests good generalisation, while a larger gap signals potential 
  overfitting.
- The mean validation RMSE, which averages the root mean squared errors for the 
  predicted changes in u-velocity (du), v-velocity (dv), and pressure (dp) 
  across all nodes. This gives a single metric to track overall predictive 
  performance.

Use this plot to assess both your model's ability to generalise to unseen data 
and the overall error in predicting next state variables. The best epoch, based 
on the lowest validation loss, is highlighted for reference.
'''
fig4 = plot_training_gap(df)
fig4.show()